In [1]:
import pandas as pd
# import utils.generate_metrics as gm
# import os 
# import warnings
# from preprocessing.process_arrow import load_arrow
# from utils.data_utils import merge_files
import utils.analysis_functions as af
from utils.data_utils import data_preprocessing
import numpy as np

In [11]:
PID_df=pd.read_csv('data/lookup/PID_location_all.csv')
csc_df= pd.read_csv('data/processed/PID_location_WSCI.csv')
npp_df=pd.read_csv('data/processed/PID_npp.csv')
df = pd.read_parquet("data/processed/dataset1.parquet")

In [13]:
df

,PID,Raos_Q,Functional_Evenness,Functional_Dispersion,Functional_Divergences,Mean Pairwise D,CHELSA_BIO_Annual_Mean_Temperature,CHELSA_BIO_Annual_Precipitation,CHELSA_BIO_Precipitation_Seasonality,CrowtherLab_SoilMoisture_intraAnnualSD_downsampled10km,...,EarthEnvTopoMed_Northness,BHAGE,managed,ownership,biome,Species Richness,Shannon Diversity,Simpson's Index,Shannon Equitabiltiy Index,source_file
0,0_42_109_1113_1,3.632738,0.697712,3.314408,0.960715,4.936123,91.617119,1373.652832,12.599973,34.799599,...,0.132662,NaN,0.0,state,Temperate broadleaf forests,6,1.101153,0.579499,0.614565,dataset16.csv
1,0_42_109_1145_1,0.417407,1.141460,1.333508,0.933902,2.380164,106.500280,1133.667480,13.700030,35.206524,...,0.097292,NaN,0.0,No Data,Temperate broadleaf forests,3,0.403497,0.196742,0.367279,dataset16.csv
2,0_42_109_1162_1,4.835461,1.037160,4.058413,0.762714,6.128560,93.567237,1373.329834,12.598328,34.466415,...,-0.631663,NaN,0.0,state,Temperate broadleaf forests,7,1.636835,0.777446,0.841167,dataset16.csv
3,0_42_109_1323_1,4.053859,0.821292,3.228536,0.917751,4.662759,106.466541,1089.783936,13.201645,36.199841,...,-0.215426,NaN,0.0,No Data,Temperate broadleaf forests,9,2.060778,0.881242,0.937900,dataset16.csv
4,0_42_109_2971_1,5.115599,0.874384,3.855560,0.849218,5.675300,107.450384,1242.034546,13.399972,36.199841,...,-0.271444,NaN,0.0,No Data,Temperate broadleaf forests,9,1.805250,0.803235,0.821604,dataset16.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98472,1_25_9_797_1,NaN,NaN,NaN,NaN,NaN,98.500000,1279.922241,9.298328,39.796814,...,0.403948,NaN,0.0,No Data,Temperate broadleaf forests,2,0.655482,0.509091,0.945660,dataset12.csv
98473,1_25_9_81_111,3.761753,0.898491,3.379716,0.862695,5.046284,90.500000,1357.086670,10.601672,40.788456,...,-0.300611,NaN,0.0,other,Temperate broadleaf forests,8,1.684693,0.776190,0.810166,dataset12.csv
98474,1_25_9_85_111,3.273756,0.929445,3.467581,0.567490,5.440403,94.483281,1299.412354,9.401671,39.657066,...,0.232948,NaN,0.0,other,Temperate broadleaf forests,6,1.409526,0.708995,0.786671,dataset12.csv
98475,1_25_9_95_1,4.851519,0.813564,3.910611,0.782025,5.490137,97.499720,1205.751465,8.900000,39.846893,...,0.393954,NaN,0.0,local,Temperate broadleaf forests,13,2.298235,0.891726,0.896016,dataset12.csv


In [3]:
def cleaning(PID_df, csc_df, npp_df):
    PID_df["managed"] = PID_df["managed"].fillna(-1)
    PID_df["ownership"] = PID_df["ownership"].fillna("No Data")
    PID_df["biome"] = PID_df["biome"].fillna("No Data")
    if 'lon' in PID_df.columns and 'lat' in PID_df.columns:
        PID_df = PID_df[PID_df["lon"] <= 0]
        PID_df = PID_df[PID_df["lat"] > 22]

    csc_df.rename(columns={'PID_left': "PID"}, inplace= True)

    csc_df=csc_df.merge(PID_df, on='PID', how='inner')

    return PID_df, csc_df, npp_df 

In [ ]:
PID_df, csc_df, npp_df = cleaning(PID_df, csc_df, npp_df)

npp_df.rename(columns={'NPP_kgC_m2_yr': "Npp"}, inplace=True)

grouped = npp_df.groupby('PID').apply(lambda x: x.sort_values('year')['Npp'].to_numpy())
grouped = grouped[grouped.apply(lambda x: ~np.isnan(x).any())]

In [6]:
grouped

PID
0_41_27_10_550       [0.8096, 0.8096, 0.8096, 0.8096, 0.8096, 0.809...
0_41_27_12_550       [0.9133, 0.9133, 0.9133, 0.9133, 0.8595, 0.859...
0_41_27_15_550       [0.8444, 0.8444, 0.8444, 0.8444, 0.7406, 0.740...
0_41_27_16_550       [0.7714000000000001, 0.7714000000000001, 0.771...
0_41_27_17_550       [0.7583000000000001, 0.7583000000000001, 0.758...
                                           ...                        
9_53_75_61977_501    [0.3784, 0.3784, 0.3784, 0.3784, 0.3784, 0.378...
9_53_75_62722_501    [0.4498, 0.4498, 0.4498, 0.4498, 0.4498, 0.462...
9_53_75_72668_501    [0.2936, 0.2936, 0.2936, 0.2936, 0.2936, 0.293...
9_53_75_93825_501    [0.4505, 0.4505, 0.4344, 0.4344, 0.4443, 0.444...
9_53_75_94177_501    [0.4171, 0.4171, 0.4171, 0.427, 0.427, 0.427, ...
Length: 401310, dtype: object

In [7]:
volatility=[]
PID_list=[]
mean=[]

for pid, arr in grouped.items():
    if arr.shape[0] > 5:
        if (arr == 0).sum() < 4: 
            v , s = af.compute_volatility(arr)
            volatility.append(v)
            PID_list.append(pid)
            mean.append(s)

npp_df = pd.DataFrame({'PID': PID_list, 'transformed npp': volatility})


In [10]:
npp_df = pd.DataFrame({'PID': PID_list, 'transformed npp': volatility})

# csc_df['csc'] = (csc_df['csc']-min(csc_df['csc']))/(max(csc_df['csc'])-min(csc_df['csc']))
# csc_df["BHAGE"] = csc_df["BHAGE"].fillna("0.0")
# csc_df=csc_df[['PID', 'csc']]

# y=['transformed npp', 'csc'] 

sd_df, fd_df=data_preprocessing(df, npp_df, csc_df)


In [14]:
fd_df.drop_duplicates(subset=['PID'], inplace=True)

In [12]:
fd_df.drop_duplicates(inplace=True)
sd_df.drop_duplicates(inplace=True)
sd_df.to_csv('sd_df.csv', index=False)
fd_df.to_csv('fd_df.csv', index=False)